In [1]:
# ============================================================
# PHASE 4.0 — FINAL MASTER TABLE + TOP GENE EXTRACTION
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)

In [2]:
# ============================================================
# PATHS
# ============================================================

import os
from pathlib import Path
import pandas as pd
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Phase 1 / 2 / 3 existing folders
PHASE1_FINAL_DIR = PROJECT_DIR / "model" / "phase1_final_protein_analysis"
PHASE2_2_DIR = PROJECT_DIR / "model" / "phase2_2_genomic_analysis"

PHASE3_DIR = PROJECT_DIR / "model" / "phase3_multimodal_integration"
PHASE3_DATA_DIR = PHASE3_DIR / "shared_dataset"
PHASE3_FEATURE_DIR = PHASE3_DIR / "features"

PHASE3_1_DIR = PHASE3_DIR / "phase3_1_modelling_light"
PHASE3_1_RESULT_DIR = PHASE3_1_DIR / "results"
PHASE3_1_MODEL_DIR = PHASE3_1_DIR / "models"

PHASE3_2_DIR = PHASE3_DIR / "phase3_2_error_modality_contribution"
PHASE3_2_RESULT_DIR = PHASE3_2_DIR / "results"

# New Phase 4 output
PHASE4_DIR = PROJECT_DIR / "model" / "phase4_biological_validation_preparation"
RESULT_DIR = PHASE4_DIR / "results"
GENE_LIST_DIR = PHASE4_DIR / "gene_lists"
REPORT_DIR = PHASE4_DIR / "reports"

for folder in [PHASE4_DIR, RESULT_DIR, GENE_LIST_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Phase 4 output:", PHASE4_DIR)

Mounted at /content/drive
Phase 4 output: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation


In [3]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


def evaluate_from_score(y_true, y_score, threshold=0.5, model_name="model"):
    y_pred = (y_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def assign_error_type(y_true, y_pred):
    if y_true == 1 and y_pred == 1:
        return "TP"
    if y_true == 0 and y_pred == 0:
        return "TN"
    if y_true == 0 and y_pred == 1:
        return "FP"
    if y_true == 1 and y_pred == 0:
        return "FN"
    return "UNKNOWN"


def safe_display_file(path):
    print("Saved:", path)

In [4]:
# ============================================================
# PHASE 4.0A — FINAL MASTER COMPARISON TABLE
# ============================================================

master_records = [
    {
        "phase": "1.1",
        "branch": "Protein",
        "representation": "AAC + Physchem",
        "embedding_policy": "handcrafted",
        "feature_type": "handcrafted protein descriptors",
        "model": "Random Forest",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.5520,
        "test_pr_auc": 0.5550,
        "test_f1": 0.5390,
        "test_mcc": 0.0480,
        "main_note": "Protein handcrafted baseline"
    },
    {
        "phase": "1.2A",
        "branch": "Protein",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "truncated_1022",
        "feature_type": "protein foundation embedding",
        "model": "Stacking",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.6202,
        "test_pr_auc": 0.6188,
        "test_f1": 0.5926,
        "test_mcc": 0.1941,
        "main_note": "ESM-2 small truncated baseline"
    },
    {
        "phase": "1.2B",
        "branch": "Protein",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "sliding_window_1022_stride_1022",
        "feature_type": "protein foundation embedding",
        "model": "Soft Voting",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.6277,
        "test_pr_auc": 0.6278,
        "test_f1": 0.5870,
        "test_mcc": 0.1650,
        "main_note": "ESM-2 small sliding-window"
    },
    {
        "phase": "1.2C",
        "branch": "Protein",
        "representation": "ESM2_t12_35M",
        "embedding_policy": "truncated_1022",
        "feature_type": "protein foundation embedding",
        "model": "Soft Voting",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.5875,
        "test_pr_auc": 0.5942,
        "test_f1": 0.5654,
        "test_mcc": 0.0995,
        "main_note": "Larger ESM-2 did not improve under truncation"
    },
    {
        "phase": "1.2D",
        "branch": "Protein",
        "representation": "ProtBERT",
        "embedding_policy": "truncated_1022",
        "feature_type": "protein foundation embedding",
        "model": "Logistic Regression",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.6371,
        "test_pr_auc": 0.6423,
        "test_f1": 0.6014,
        "test_mcc": 0.1943,
        "main_note": "ProtBERT truncated"
    },
    {
        "phase": "1.2E",
        "branch": "Protein",
        "representation": "ProtBERT",
        "embedding_policy": "sliding_window_1022_stride_1022",
        "feature_type": "protein foundation embedding",
        "model": "Logistic Regression",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.6487,
        "test_pr_auc": 0.6551,
        "test_f1": 0.5896,
        "test_mcc": 0.1941,
        "main_note": "Best protein ranking representation before shared split"
    },
    {
        "phase": "2.1",
        "branch": "Genomic regulatory",
        "representation": "K3 + K4 + Basic",
        "embedding_policy": "handcrafted TSS-proximal 2kbUp+500bpDown",
        "feature_type": "genomic handcrafted k-mer + GC/CpG features",
        "model": "Random Forest",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.6496,
        "test_pr_auc": 0.6327,
        "test_f1": 0.6406,
        "test_mcc": 0.2557,
        "main_note": "Official genomic handcrafted model"
    },
    {
        "phase": "2.2",
        "branch": "Genomic regulatory",
        "representation": "K3 + K4 + Basic",
        "embedding_policy": "handcrafted TSS-proximal 2kbUp+500bpDown",
        "feature_type": "genomic handcrafted k-mer + GC/CpG features",
        "model": "SVM RBF",
        "selection_policy": "diagnostic test comparison",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.6765,
        "test_pr_auc": 0.6504,
        "test_f1": 0.6620,
        "test_mcc": 0.2933,
        "main_note": "Diagnostic best genomic model, not official selection"
    },
    {
        "phase": "3.1",
        "branch": "Multimodal",
        "representation": "ProtBERT SW + K3/K4/Basic",
        "embedding_policy": "early fusion direct concatenation",
        "feature_type": "protein embedding + genomic handcrafted",
        "model": "SVM RBF",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.7290,
        "test_pr_auc": 0.7573,
        "test_f1": 0.6590,
        "test_mcc": 0.3438,
        "main_note": "Official multimodal integration model"
    },
    {
        "phase": "3.1",
        "branch": "Protein shared split",
        "representation": "ProtBERT SW",
        "embedding_policy": "shared split",
        "feature_type": "protein foundation embedding",
        "model": "SVM RBF",
        "selection_policy": "validation ROC-AUC",
        "threshold_policy": "default_0.5",
        "test_roc_auc": 0.7274,
        "test_pr_auc": 0.7433,
        "test_f1": 0.6667,
        "test_mcc": 0.3215,
        "main_note": "Best protein-only model on shared Phase 3 split"
    }
]

master_comparison_df = pd.DataFrame(master_records)

master_comparison_df = master_comparison_df.sort_values(
    by=["test_roc_auc", "test_pr_auc", "test_mcc"],
    ascending=False
).reset_index(drop=True)

display(master_comparison_df)

master_path = RESULT_DIR / "phase4_0A_final_master_comparison_table.csv"
master_comparison_df.to_csv(master_path, index=False)
safe_display_file(master_path)

,phase,branch,representation,embedding_policy,feature_type,model,selection_policy,threshold_policy,test_roc_auc,test_pr_auc,test_f1,test_mcc,main_note
0,3.1,Multimodal,ProtBERT SW + K3/K4/Basic,early fusion direct concatenation,protein embedding + genomic handcrafted,SVM RBF,validation ROC-AUC,default_0.5,0.7290,0.7573,0.6590,0.3438,Official multimodal integration model
1,3.1,Protein shared split,ProtBERT SW,shared split,protein foundation embedding,SVM RBF,validation ROC-AUC,default_0.5,0.7274,0.7433,0.6667,0.3215,Best protein-only model on shared Phase 3 split
2,2.2,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,SVM RBF,diagnostic test comparison,default_0.5,0.6765,0.6504,0.6620,0.2933,"Diagnostic best genomic model, not official selection"
3,2.1,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,Random Forest,validation ROC-AUC,default_0.5,0.6496,0.6327,0.6406,0.2557,Official genomic handcrafted model
4,1.2E,Protein,ProtBERT,sliding_window_1022_stride_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6487,0.6551,0.5896,0.1941,Best protein ranking representation before shared split
5,1.2D,Protein,ProtBERT,truncated_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6371,0.6423,0.6014,0.1943,ProtBERT truncated
6,1.2B,Protein,ESM2_t6_8M,sliding_window_1022_stride_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.6277,0.6278,0.5870,0.1650,ESM-2 small sliding-window
7,1.2A,Protein,ESM2_t6_8M,truncated_1022,protein foundation embedding,Stacking,validation ROC-AUC,default_0.5,0.6202,0.6188,0.5926,0.1941,ESM-2 small truncated baseline
8,1.2C,Protein,ESM2_t12_35M,truncated_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.5875,0.5942,0.5654,0.0995,Larger ESM-2 did not improve under truncation
9,1.1,Protein,AAC + Physchem,handcrafted,handcrafted protein descriptors,Random Forest,validation ROC-AUC,default_0.5,0.5520,0.5550,0.5390,0.0480,Protein handcrafted baseline


Saved: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0A_final_master_comparison_table.csv


In [5]:
# ============================================================
# MASTER RANKING TABLES
# ============================================================

roc_ranking_df = master_comparison_df.sort_values(
    by="test_roc_auc",
    ascending=False
)[[
    "phase",
    "branch",
    "representation",
    "model",
    "test_roc_auc",
    "test_pr_auc",
    "test_f1",
    "test_mcc",
    "main_note"
]]

pr_ranking_df = master_comparison_df.sort_values(
    by="test_pr_auc",
    ascending=False
)[[
    "phase",
    "branch",
    "representation",
    "model",
    "test_roc_auc",
    "test_pr_auc",
    "test_f1",
    "test_mcc",
    "main_note"
]]

mcc_ranking_df = master_comparison_df.sort_values(
    by="test_mcc",
    ascending=False
)[[
    "phase",
    "branch",
    "representation",
    "model",
    "test_roc_auc",
    "test_pr_auc",
    "test_f1",
    "test_mcc",
    "main_note"
]]

print("ROC-AUC ranking")
display(roc_ranking_df)

print("PR-AUC ranking")
display(pr_ranking_df)

print("MCC ranking")
display(mcc_ranking_df)

roc_ranking_df.to_csv(RESULT_DIR / "phase4_0A_ranking_by_roc_auc.csv", index=False)
pr_ranking_df.to_csv(RESULT_DIR / "phase4_0A_ranking_by_pr_auc.csv", index=False)
mcc_ranking_df.to_csv(RESULT_DIR / "phase4_0A_ranking_by_mcc.csv", index=False)

ROC-AUC ranking


,phase,branch,representation,model,test_roc_auc,test_pr_auc,test_f1,test_mcc,main_note
0,3.1,Multimodal,ProtBERT SW + K3/K4/Basic,SVM RBF,0.7290,0.7573,0.6590,0.3438,Official multimodal integration model
1,3.1,Protein shared split,ProtBERT SW,SVM RBF,0.7274,0.7433,0.6667,0.3215,Best protein-only model on shared Phase 3 split
2,2.2,Genomic regulatory,K3 + K4 + Basic,SVM RBF,0.6765,0.6504,0.6620,0.2933,"Diagnostic best genomic model, not official selection"
3,2.1,Genomic regulatory,K3 + K4 + Basic,Random Forest,0.6496,0.6327,0.6406,0.2557,Official genomic handcrafted model
4,1.2E,Protein,ProtBERT,Logistic Regression,0.6487,0.6551,0.5896,0.1941,Best protein ranking representation before shared split
5,1.2D,Protein,ProtBERT,Logistic Regression,0.6371,0.6423,0.6014,0.1943,ProtBERT truncated
6,1.2B,Protein,ESM2_t6_8M,Soft Voting,0.6277,0.6278,0.5870,0.1650,ESM-2 small sliding-window
7,1.2A,Protein,ESM2_t6_8M,Stacking,0.6202,0.6188,0.5926,0.1941,ESM-2 small truncated baseline
8,1.2C,Protein,ESM2_t12_35M,Soft Voting,0.5875,0.5942,0.5654,0.0995,Larger ESM-2 did not improve under truncation
9,1.1,Protein,AAC + Physchem,Random Forest,0.5520,0.5550,0.5390,0.0480,Protein handcrafted baseline


PR-AUC ranking


,phase,branch,representation,model,test_roc_auc,test_pr_auc,test_f1,test_mcc,main_note
0,3.1,Multimodal,ProtBERT SW + K3/K4/Basic,SVM RBF,0.7290,0.7573,0.6590,0.3438,Official multimodal integration model
1,3.1,Protein shared split,ProtBERT SW,SVM RBF,0.7274,0.7433,0.6667,0.3215,Best protein-only model on shared Phase 3 split
4,1.2E,Protein,ProtBERT,Logistic Regression,0.6487,0.6551,0.5896,0.1941,Best protein ranking representation before shared split
2,2.2,Genomic regulatory,K3 + K4 + Basic,SVM RBF,0.6765,0.6504,0.6620,0.2933,"Diagnostic best genomic model, not official selection"
5,1.2D,Protein,ProtBERT,Logistic Regression,0.6371,0.6423,0.6014,0.1943,ProtBERT truncated
3,2.1,Genomic regulatory,K3 + K4 + Basic,Random Forest,0.6496,0.6327,0.6406,0.2557,Official genomic handcrafted model
6,1.2B,Protein,ESM2_t6_8M,Soft Voting,0.6277,0.6278,0.5870,0.1650,ESM-2 small sliding-window
7,1.2A,Protein,ESM2_t6_8M,Stacking,0.6202,0.6188,0.5926,0.1941,ESM-2 small truncated baseline
8,1.2C,Protein,ESM2_t12_35M,Soft Voting,0.5875,0.5942,0.5654,0.0995,Larger ESM-2 did not improve under truncation
9,1.1,Protein,AAC + Physchem,Random Forest,0.5520,0.5550,0.5390,0.0480,Protein handcrafted baseline


MCC ranking


,phase,branch,representation,model,test_roc_auc,test_pr_auc,test_f1,test_mcc,main_note
0,3.1,Multimodal,ProtBERT SW + K3/K4/Basic,SVM RBF,0.7290,0.7573,0.6590,0.3438,Official multimodal integration model
1,3.1,Protein shared split,ProtBERT SW,SVM RBF,0.7274,0.7433,0.6667,0.3215,Best protein-only model on shared Phase 3 split
2,2.2,Genomic regulatory,K3 + K4 + Basic,SVM RBF,0.6765,0.6504,0.6620,0.2933,"Diagnostic best genomic model, not official selection"
3,2.1,Genomic regulatory,K3 + K4 + Basic,Random Forest,0.6496,0.6327,0.6406,0.2557,Official genomic handcrafted model
5,1.2D,Protein,ProtBERT,Logistic Regression,0.6371,0.6423,0.6014,0.1943,ProtBERT truncated
4,1.2E,Protein,ProtBERT,Logistic Regression,0.6487,0.6551,0.5896,0.1941,Best protein ranking representation before shared split
7,1.2A,Protein,ESM2_t6_8M,Stacking,0.6202,0.6188,0.5926,0.1941,ESM-2 small truncated baseline
6,1.2B,Protein,ESM2_t6_8M,Soft Voting,0.6277,0.6278,0.5870,0.1650,ESM-2 small sliding-window
8,1.2C,Protein,ESM2_t12_35M,Soft Voting,0.5875,0.5942,0.5654,0.0995,Larger ESM-2 did not improve under truncation
9,1.1,Protein,AAC + Physchem,Random Forest,0.5520,0.5550,0.5390,0.0480,Protein handcrafted baseline


In [6]:
# ============================================================
# PHASE 4.0B — LOAD OFFICIAL COMBINED SVM + SHARED TEST DATA
# ============================================================

official_model_path = PHASE3_1_MODEL_DIR / "phase3_1_combined_proteinplusgenomic_svm_rbf_light.pkl"

print("Official model exists:", official_model_path.exists(), official_model_path)
assert official_model_path.exists(), "Official Combined SVM model file not found."

official_combined_model = joblib.load(official_model_path)

# Shared test data
test_meta = pd.read_csv(PHASE3_DATA_DIR / "test_multimodal_metadata_v1.csv")
y_test = np.load(PHASE3_DATA_DIR / "y_test_multimodal_v1.npy")
X_test_combined = np.load(PHASE3_FEATURE_DIR / "X_test_combined_protein_genomic_v1.npy")

print("test_meta:", test_meta.shape)
print("y_test:", y_test.shape)
print("X_test_combined:", X_test_combined.shape)

display(test_meta.head())

Official model exists: True /content/drive/MyDrive/Project_Protein/model/phase3_multimodal_integration/phase3_1_modelling_light/models/phase3_1_combined_proteinplusgenomic_svm_rbf_light.pkl
test_meta: (271, 12)
y_test: (271,)
X_test_combined: (271, 1380)


,gene_id,gene_symbol_protein,label_protein,protein_row_index,original_protein_split,gene_symbol_genomic,label_genomic,genomic_row_index,original_genomic_split,label_match,label,gene_symbol
0,ENSG00000122971,ACADS,1,593,train,ACADS,1,1151,train,True,1,ACADS
1,ENSG00000162104,ADCY9,1,693,train,ADCY9,1,399,train,True,1,ADCY9
2,ENSG00000123146,ADGRE5,0,115,train,ADGRE5,0,880,train,True,0,ADGRE5
3,ENSG00000172594,SMPDL3A,0,1609,test,SMPDL3A,0,617,train,True,0,SMPDL3A
4,ENSG00000163116,STPG2,0,159,train,STPG2,0,207,train,True,0,STPG2


In [7]:
# ============================================================
# GENERATE OFFICIAL COMBINED SVM TEST PREDICTIONS
# ============================================================

combined_score = get_positive_class_score(official_combined_model, X_test_combined)
combined_pred = (combined_score >= 0.5).astype(int)

combined_test_predictions_df = test_meta.copy()

combined_test_predictions_df["true_label"] = y_test
combined_test_predictions_df["combined_svm_score"] = combined_score
combined_test_predictions_df["combined_svm_pred"] = combined_pred
combined_test_predictions_df["combined_error_type"] = [
    assign_error_type(y_true, y_pred)
    for y_true, y_pred in zip(y_test, combined_pred)
]

combined_test_predictions_df["combined_correct"] = (
    combined_test_predictions_df["true_label"] ==
    combined_test_predictions_df["combined_svm_pred"]
)

combined_test_predictions_df = combined_test_predictions_df.sort_values(
    by="combined_svm_score",
    ascending=False
).reset_index(drop=True)

display(combined_test_predictions_df.head(20))

pred_path = RESULT_DIR / "phase4_0B_official_combined_svm_all_test_predictions.csv"
combined_test_predictions_df.to_csv(pred_path, index=False)
safe_display_file(pred_path)

,gene_id,gene_symbol_protein,label_protein,protein_row_index,original_protein_split,gene_symbol_genomic,label_genomic,genomic_row_index,original_genomic_split,label_match,label,gene_symbol,true_label,combined_svm_score,combined_svm_pred,combined_error_type,combined_correct
0,ENSG00000053254,FOXN3,1,671,train,FOXN3,1,193,train,True,1,FOXN3,1,0.884630,1,TP,True
1,ENSG00000276644,DACH1,1,549,train,DACH1,1,919,train,True,1,DACH1,1,0.841229,1,TP,True
2,ENSG00000090266,NDUFB2,1,1719,test,NDUFB2,1,930,train,True,1,NDUFB2,1,0.831156,1,TP,True
3,ENSG00000006468,ETV1,1,1701,test,ETV1,1,540,train,True,1,ETV1,1,0.824098,1,TP,True
4,ENSG00000162992,NEUROD1,1,1726,test,NEUROD1,1,795,train,True,1,NEUROD1,1,0.818462,1,TP,True
5,ENSG00000171105,INSR,1,256,train,INSR,1,939,train,True,1,INSR,1,0.817504,1,TP,True
6,ENSG00000128683,GAD1,1,958,train,GAD1,1,1452,validation,True,1,GAD1,1,0.813424,1,TP,True
7,ENSG00000107864,CPEB3,1,1430,validation,CPEB3,1,968,train,True,1,CPEB3,1,0.808824,1,TP,True
8,ENSG00000121297,TSHZ3,1,1324,validation,TSHZ3,1,1741,test,True,1,TSHZ3,1,0.808711,1,TP,True
9,ENSG00000127334,DYRK2,1,1040,train,DYRK2,1,532,train,True,1,DYRK2,1,0.805801,1,TP,True


Saved: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_official_combined_svm_all_test_predictions.csv


In [8]:
# ============================================================
# VERIFY OFFICIAL MODEL METRICS
# ============================================================

official_metrics = evaluate_from_score(
    y_true=y_test,
    y_score=combined_score,
    threshold=0.5,
    model_name="Official Combined Protein+Genomic SVM RBF"
)

official_metrics_df = pd.DataFrame([official_metrics])

display(official_metrics_df)

official_metrics_path = RESULT_DIR / "phase4_0B_official_combined_svm_metric_check.csv"
official_metrics_df.to_csv(official_metrics_path, index=False)
safe_display_file(official_metrics_path)

,model,threshold,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Official Combined Protein+Genomic SVM RBF,0.5,0.671587,0.68254,0.637037,0.705882,0.659004,0.728976,0.757253,0.343763,96,40,49,86


Saved: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_official_combined_svm_metric_check.csv


In [9]:
# ============================================================
# TOP-RANKED PREDICTED T2D-ASSOCIATED GENES
# ============================================================

top_n = 50

top_50_predicted_t2d_df = combined_test_predictions_df.sort_values(
    by="combined_svm_score",
    ascending=False
).head(top_n).copy()

display(top_50_predicted_t2d_df[[
    "gene_id",
    "gene_symbol",
    "true_label",
    "combined_svm_score",
    "combined_svm_pred",
    "combined_error_type"
]].head(50))

top50_path = GENE_LIST_DIR / "phase4_0B_top_50_combined_svm_predicted_t2d_genes.csv"
top_50_predicted_t2d_df.to_csv(top50_path, index=False)
safe_display_file(top50_path)

,gene_id,gene_symbol,true_label,combined_svm_score,combined_svm_pred,combined_error_type
0,ENSG00000053254,FOXN3,1,0.884630,1,TP
1,ENSG00000276644,DACH1,1,0.841229,1,TP
2,ENSG00000090266,NDUFB2,1,0.831156,1,TP
3,ENSG00000006468,ETV1,1,0.824098,1,TP
4,ENSG00000162992,NEUROD1,1,0.818462,1,TP
5,ENSG00000171105,INSR,1,0.817504,1,TP
6,ENSG00000128683,GAD1,1,0.813424,1,TP
7,ENSG00000107864,CPEB3,1,0.808824,1,TP
8,ENSG00000121297,TSHZ3,1,0.808711,1,TP
9,ENSG00000127334,DYRK2,1,0.805801,1,TP


Saved: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/gene_lists/phase4_0B_top_50_combined_svm_predicted_t2d_genes.csv


In [10]:
# ============================================================
# EXTRACT TP / FP / FN / TN GENE LISTS
# ============================================================

top_true_positives_df = combined_test_predictions_df[
    combined_test_predictions_df["combined_error_type"] == "TP"
].sort_values(
    by="combined_svm_score",
    ascending=False
).copy()

high_confidence_false_positives_df = combined_test_predictions_df[
    combined_test_predictions_df["combined_error_type"] == "FP"
].sort_values(
    by="combined_svm_score",
    ascending=False
).copy()

high_confidence_false_negatives_df = combined_test_predictions_df[
    combined_test_predictions_df["combined_error_type"] == "FN"
].sort_values(
    by="combined_svm_score",
    ascending=True
).copy()

top_true_negatives_df = combined_test_predictions_df[
    combined_test_predictions_df["combined_error_type"] == "TN"
].sort_values(
    by="combined_svm_score",
    ascending=True
).copy()

print("Top true positives:", top_true_positives_df.shape)
display(top_true_positives_df[[
    "gene_id", "gene_symbol", "true_label", "combined_svm_score", "combined_error_type"
]].head(30))

print("High-confidence false positives:", high_confidence_false_positives_df.shape)
display(high_confidence_false_positives_df[[
    "gene_id", "gene_symbol", "true_label", "combined_svm_score", "combined_error_type"
]].head(30))

print("High-confidence false negatives:", high_confidence_false_negatives_df.shape)
display(high_confidence_false_negatives_df[[
    "gene_id", "gene_symbol", "true_label", "combined_svm_score", "combined_error_type"
]].head(30))

print("Top true negatives:", top_true_negatives_df.shape)
display(top_true_negatives_df[[
    "gene_id", "gene_symbol", "true_label", "combined_svm_score", "combined_error_type"
]].head(30))

top_true_positives_df.to_csv(GENE_LIST_DIR / "phase4_0B_top_true_positives_combined_svm.csv", index=False)
high_confidence_false_positives_df.to_csv(GENE_LIST_DIR / "phase4_0B_high_confidence_false_positives_combined_svm.csv", index=False)
high_confidence_false_negatives_df.to_csv(GENE_LIST_DIR / "phase4_0B_high_confidence_false_negatives_combined_svm.csv", index=False)
top_true_negatives_df.to_csv(GENE_LIST_DIR / "phase4_0B_top_true_negatives_combined_svm.csv", index=False)

Top true positives: (86, 17)


,gene_id,gene_symbol,true_label,combined_svm_score,combined_error_type
0,ENSG00000053254,FOXN3,1,0.884630,TP
1,ENSG00000276644,DACH1,1,0.841229,TP
2,ENSG00000090266,NDUFB2,1,0.831156,TP
3,ENSG00000006468,ETV1,1,0.824098,TP
4,ENSG00000162992,NEUROD1,1,0.818462,TP
5,ENSG00000171105,INSR,1,0.817504,TP
6,ENSG00000128683,GAD1,1,0.813424,TP
7,ENSG00000107864,CPEB3,1,0.808824,TP
8,ENSG00000121297,TSHZ3,1,0.808711,TP
9,ENSG00000127334,DYRK2,1,0.805801,TP


High-confidence false positives: (40, 17)


,gene_id,gene_symbol,true_label,combined_svm_score,combined_error_type
22,ENSG00000160221,GATD3,0,0.770886,FP
26,ENSG00000141314,RHBDL3,0,0.761844,FP
34,ENSG00000108091,CCDC6,0,0.742967,FP
37,ENSG00000196277,GRM7,0,0.732043,FP
40,ENSG00000157212,PAXIP1,0,0.723585,FP
41,ENSG00000135930,EIF4E2,0,0.718188,FP
44,ENSG00000146267,FAXC,0,0.711850,FP
54,ENSG00000105135,ILVBL,0,0.692967,FP
56,ENSG00000104833,TUBB4A,0,0.691734,FP
58,ENSG00000172594,SMPDL3A,0,0.690366,FP


High-confidence false negatives: (49, 17)


,gene_id,gene_symbol,true_label,combined_svm_score,combined_error_type
269,ENSG00000103932,RPAP1,1,0.118474,FN
259,ENSG00000184730,APOBR,1,0.187996,FN
256,ENSG00000104976,SNAPC2,1,0.197375,FN
253,ENSG00000108557,RAI1,1,0.203728,FN
252,ENSG00000115970,THADA,1,0.208333,FN
249,ENSG00000187240,DYNC2H1,1,0.217003,FN
244,ENSG00000164530,PI16,1,0.232008,FN
243,ENSG00000002822,MAD1L1,1,0.234614,FN
241,ENSG00000107201,RIGI,1,0.235891,FN
228,ENSG00000164935,DCSTAMP,1,0.274267,FN


Top true negatives: (96, 17)


,gene_id,gene_symbol,true_label,combined_svm_score,combined_error_type
270,ENSG00000286038,SPDYE13,0,0.089823,TN
268,ENSG00000169515,CCDC8,0,0.130646,TN
267,ENSG00000179044,EXOC3L1,0,0.145096,TN
266,ENSG00000154767,XPC,0,0.158756,TN
265,ENSG00000196110,ZNF699,0,0.163819,TN
264,ENSG00000166317,SYNPO2L,0,0.167729,TN
263,ENSG00000128335,APOL2,0,0.171959,TN
262,ENSG00000159625,DRC7,0,0.173342,TN
261,ENSG00000132972,RNF17,0,0.178679,TN
260,ENSG00000143337,TOR1AIP1,0,0.186369,TN


In [11]:
# ============================================================
# LOAD PHASE 3.2 ERROR COMPARISON GROUPS
# ============================================================

phase3_2_comparison_path = PHASE3_2_RESULT_DIR / "phase3_2A_protein_vs_combined_svm_test_prediction_comparison.csv"

print("Phase 3.2 comparison exists:", phase3_2_comparison_path.exists(), phase3_2_comparison_path)

if phase3_2_comparison_path.exists():
    phase3_2_comparison_df = pd.read_csv(phase3_2_comparison_path)
    print("Loaded:", phase3_2_comparison_df.shape)
    display(phase3_2_comparison_df.head())
else:
    phase3_2_comparison_df = None
    print("Phase 3.2 comparison file not found. Skipping rescued/lost gene extraction.")

Phase 3.2 comparison exists: True /content/drive/MyDrive/Project_Protein/model/phase3_multimodal_integration/phase3_2_error_modality_contribution/results/phase3_2A_protein_vs_combined_svm_test_prediction_comparison.csv
Loaded: (271, 14)


,gene_id,gene_symbol,label,true_label,protein_svm_score,combined_svm_score,protein_svm_pred,combined_svm_pred,protein_correct,combined_correct,protein_error_type,combined_error_type,score_delta_combined_minus_protein,integration_error_group
0,ENSG00000122971,ACADS,1,1,0.719627,0.750810,1,1,True,True,TP,TP,0.031183,both_correct
1,ENSG00000162104,ADCY9,1,1,0.680303,0.756812,1,1,True,True,TP,TP,0.076509,both_correct
2,ENSG00000123146,ADGRE5,0,0,0.554474,0.532796,1,1,False,False,FP,FP,-0.021678,both_wrong
3,ENSG00000172594,SMPDL3A,0,0,0.683032,0.690366,1,1,False,False,FP,FP,0.007335,both_wrong
4,ENSG00000163116,STPG2,0,0,0.237920,0.235603,0,0,True,True,TN,TN,-0.002317,both_correct


In [12]:
# ============================================================
# EXTRACT COMBINED-RESCUED AND LOST GENES
# ============================================================

if phase3_2_comparison_df is not None:

    combined_rescued_genes_df = phase3_2_comparison_df[
        phase3_2_comparison_df["integration_error_group"] == "combined_correct_protein_wrong"
    ].copy()

    protein_lost_after_integration_df = phase3_2_comparison_df[
        phase3_2_comparison_df["integration_error_group"] == "protein_correct_combined_wrong"
    ].copy()

    both_wrong_genes_df = phase3_2_comparison_df[
        phase3_2_comparison_df["integration_error_group"] == "both_wrong"
    ].copy()

    both_correct_genes_df = phase3_2_comparison_df[
        phase3_2_comparison_df["integration_error_group"] == "both_correct"
    ].copy()

    combined_rescued_genes_df = combined_rescued_genes_df.sort_values(
        by=["true_label", "combined_svm_score"],
        ascending=[False, False]
    )

    protein_lost_after_integration_df = protein_lost_after_integration_df.sort_values(
        by=["true_label", "protein_svm_score"],
        ascending=[False, False]
    )

    print("Combined rescued genes:", combined_rescued_genes_df.shape)
    display(combined_rescued_genes_df[[
        "gene_id",
        "gene_symbol",
        "true_label",
        "protein_svm_score",
        "combined_svm_score",
        "protein_error_type",
        "combined_error_type",
        "score_delta_combined_minus_protein"
    ]].head(50))

    print("Protein-correct but lost after integration:", protein_lost_after_integration_df.shape)
    display(protein_lost_after_integration_df[[
        "gene_id",
        "gene_symbol",
        "true_label",
        "protein_svm_score",
        "combined_svm_score",
        "protein_error_type",
        "combined_error_type",
        "score_delta_combined_minus_protein"
    ]].head(50))

    combined_rescued_genes_df.to_csv(
        GENE_LIST_DIR / "phase4_0B_combined_rescued_genes.csv",
        index=False
    )

    protein_lost_after_integration_df.to_csv(
        GENE_LIST_DIR / "phase4_0B_protein_correct_but_lost_after_integration_genes.csv",
        index=False
    )

    both_wrong_genes_df.to_csv(
        GENE_LIST_DIR / "phase4_0B_both_wrong_genes.csv",
        index=False
    )

    both_correct_genes_df.to_csv(
        GENE_LIST_DIR / "phase4_0B_both_correct_genes.csv",
        index=False
    )

Combined rescued genes: (18, 14)


,gene_id,gene_symbol,true_label,protein_svm_score,combined_svm_score,protein_error_type,combined_error_type,score_delta_combined_minus_protein
115,ENSG00000151883,PARP8,1,0.457089,0.540746,FN,TP,0.083657
112,ENSG00000006071,ABCC8,1,0.469028,0.522492,FN,TP,0.053464
29,ENSG00000134640,MTNR1B,1,0.494592,0.500000,FN,TP,0.005408
76,ENSG00000103978,TMEM87A,0,0.617864,0.494235,FP,TN,-0.123629
54,ENSG00000122550,KLHL7,0,0.714343,0.491790,FP,TN,-0.222553
229,ENSG00000105738,SIPA1L3,0,0.597829,0.486927,FP,TN,-0.110902
94,ENSG00000077327,SPAG6,0,0.556301,0.476395,FP,TN,-0.079906
99,ENSG00000197530,MIB2,0,0.576509,0.463581,FP,TN,-0.112928
107,ENSG00000154832,CXXC1,0,0.607166,0.460217,FP,TN,-0.146949
6,ENSG00000173320,STOX2,0,0.569132,0.452565,FP,TN,-0.116567


Protein-correct but lost after integration: (15, 14)


,gene_id,gene_symbol,true_label,protein_svm_score,combined_svm_score,protein_error_type,combined_error_type,score_delta_combined_minus_protein
10,ENSG00000156219,ART3,1,0.619308,0.352695,TP,FN,-0.266613
24,ENSG00000101391,CDK5RAP1,1,0.607145,0.494782,TP,FN,-0.112363
113,ENSG00000134480,CCNH,1,0.585092,0.359140,TP,FN,-0.225952
53,ENSG00000241878,PISD,1,0.578919,0.441545,TP,FN,-0.137375
209,ENSG00000185345,PRKN,1,0.568179,0.441144,TP,FN,-0.127035
153,ENSG00000147684,NDUFB9,1,0.531210,0.461754,TP,FN,-0.069456
127,ENSG00000128805,ARHGAP22,1,0.529745,0.435654,TP,FN,-0.094091
188,ENSG00000165623,UCMA,1,0.517002,0.492378,TP,FN,-0.024624
83,ENSG00000131778,CHD1L,1,0.507651,0.491878,TP,FN,-0.015772
12,ENSG00000157212,PAXIP1,0,0.487344,0.723585,TN,FP,0.236241


In [13]:
# ============================================================
# EXPORT CLEAN GENE SYMBOL LISTS FOR ENRICHMENT
# ============================================================

def export_gene_symbol_list(df, output_txt_path, symbol_col="gene_symbol"):
    gene_symbols = (
        df[symbol_col]
        .dropna()
        .astype(str)
        .str.strip()
    )

    gene_symbols = gene_symbols[gene_symbols != ""]
    gene_symbols = gene_symbols.drop_duplicates().tolist()

    with open(output_txt_path, "w") as f:
        for gene in gene_symbols:
            f.write(gene + "\n")

    print("Saved gene list:", output_txt_path, "| n =", len(gene_symbols))

    return gene_symbols


# Main gene lists
top50_symbols = export_gene_symbol_list(
    top_50_predicted_t2d_df,
    GENE_LIST_DIR / "gene_symbols_top_50_combined_predictions.txt"
)

tp_symbols = export_gene_symbol_list(
    top_true_positives_df,
    GENE_LIST_DIR / "gene_symbols_top_true_positives.txt"
)

fp_symbols = export_gene_symbol_list(
    high_confidence_false_positives_df,
    GENE_LIST_DIR / "gene_symbols_high_confidence_false_positives.txt"
)

fn_symbols = export_gene_symbol_list(
    high_confidence_false_negatives_df,
    GENE_LIST_DIR / "gene_symbols_high_confidence_false_negatives.txt"
)

if phase3_2_comparison_df is not None:
    rescued_symbols = export_gene_symbol_list(
        combined_rescued_genes_df,
        GENE_LIST_DIR / "gene_symbols_combined_rescued_genes.txt"
    )

    both_wrong_symbols = export_gene_symbol_list(
        both_wrong_genes_df,
        GENE_LIST_DIR / "gene_symbols_both_wrong_genes.txt"
    )

Saved gene list: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/gene_lists/gene_symbols_top_50_combined_predictions.txt | n = 50
Saved gene list: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/gene_lists/gene_symbols_top_true_positives.txt | n = 86
Saved gene list: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/gene_lists/gene_symbols_high_confidence_false_positives.txt | n = 40
Saved gene list: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/gene_lists/gene_symbols_high_confidence_false_negatives.txt | n = 49
Saved gene list: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/gene_lists/gene_symbols_combined_rescued_genes.txt | n = 18
Saved gene list: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/gene_lists/gene_symbols_both_wrong_genes.txt | n = 74


In [14]:
# ============================================================
# GENE LIST SUMMARY TABLE
# ============================================================

gene_list_summary_records = [
    {
        "gene_list_name": "top_50_combined_predictions",
        "n_genes": len(top50_symbols),
        "description": "Top 50 genes ranked by official Combined SVM predicted T2D score"
    },
    {
        "gene_list_name": "top_true_positives",
        "n_genes": len(tp_symbols),
        "description": "Correctly predicted T2D-associated genes ranked by score"
    },
    {
        "gene_list_name": "high_confidence_false_positives",
        "n_genes": len(fp_symbols),
        "description": "Background genes predicted as T2D-associated with high confidence"
    },
    {
        "gene_list_name": "high_confidence_false_negatives",
        "n_genes": len(fn_symbols),
        "description": "T2D-associated genes predicted as background"
    }
]

if phase3_2_comparison_df is not None:
    gene_list_summary_records.extend([
        {
            "gene_list_name": "combined_rescued_genes",
            "n_genes": len(rescued_symbols),
            "description": "Genes correctly predicted by Combined SVM but incorrectly predicted by Protein-only SVM"
        },
        {
            "gene_list_name": "both_wrong_genes",
            "n_genes": len(both_wrong_symbols),
            "description": "Genes misclassified by both Protein-only SVM and Combined SVM"
        }
    ])

gene_list_summary_df = pd.DataFrame(gene_list_summary_records)

display(gene_list_summary_df)

gene_list_summary_path = RESULT_DIR / "phase4_0B_gene_list_summary.csv"
gene_list_summary_df.to_csv(gene_list_summary_path, index=False)
safe_display_file(gene_list_summary_path)

,gene_list_name,n_genes,description
0,top_50_combined_predictions,50,Top 50 genes ranked by official Combined SVM predicted T2D score
1,top_true_positives,86,Correctly predicted T2D-associated genes ranked by score
2,high_confidence_false_positives,40,Background genes predicted as T2D-associated with high confidence
3,high_confidence_false_negatives,49,T2D-associated genes predicted as background
4,combined_rescued_genes,18,Genes correctly predicted by Combined SVM but incorrectly predicted by Protein-only SVM
5,both_wrong_genes,74,Genes misclassified by both Protein-only SVM and Combined SVM


Saved: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_gene_list_summary.csv


In [15]:
# ============================================================
# ADD BIOLOGICAL VALIDATION PRIORITY LABELS
# ============================================================

bio_validation_candidates_df = combined_test_predictions_df.copy()

bio_validation_candidates_df["rank_by_combined_score"] = (
    bio_validation_candidates_df["combined_svm_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

def assign_validation_priority(row):
    if row["combined_error_type"] == "TP" and row["rank_by_combined_score"] <= 50:
        return "top_ranked_true_positive"
    if row["combined_error_type"] == "FP" and row["combined_svm_score"] >= 0.70:
        return "high_confidence_false_positive"
    if row["combined_error_type"] == "FN" and row["combined_svm_score"] <= 0.30:
        return "high_confidence_false_negative"
    if row["rank_by_combined_score"] <= 50:
        return "top_50_prediction"
    return "standard"

bio_validation_candidates_df["validation_priority"] = bio_validation_candidates_df.apply(
    assign_validation_priority,
    axis=1
)

display(
    bio_validation_candidates_df[[
        "gene_id",
        "gene_symbol",
        "true_label",
        "combined_svm_score",
        "combined_svm_pred",
        "combined_error_type",
        "rank_by_combined_score",
        "validation_priority"
    ]].head(80)
)

bio_priority_path = RESULT_DIR / "phase4_0B_biological_validation_candidate_table.csv"
bio_validation_candidates_df.to_csv(bio_priority_path, index=False)
safe_display_file(bio_priority_path)

,gene_id,gene_symbol,true_label,combined_svm_score,combined_svm_pred,combined_error_type,rank_by_combined_score,validation_priority
0,ENSG00000053254,FOXN3,1,0.884630,1,TP,1,top_ranked_true_positive
1,ENSG00000276644,DACH1,1,0.841229,1,TP,2,top_ranked_true_positive
2,ENSG00000090266,NDUFB2,1,0.831156,1,TP,3,top_ranked_true_positive
3,ENSG00000006468,ETV1,1,0.824098,1,TP,4,top_ranked_true_positive
4,ENSG00000162992,NEUROD1,1,0.818462,1,TP,5,top_ranked_true_positive
...,...,...,...,...,...,...,...,...
75,ENSG00000154269,ENPP3,1,0.645141,1,TP,76,standard
76,ENSG00000132437,DDC,1,0.645004,1,TP,77,standard
77,ENSG00000077044,DGKD,1,0.640977,1,TP,78,standard
78,ENSG00000065183,WDR3,0,0.640898,1,FP,79,standard


Saved: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_biological_validation_candidate_table.csv


In [16]:
# ============================================================
# WRITE PHASE 4.0 PREPARATION SUMMARY REPORT
# ============================================================

n_top50 = len(top_50_predicted_t2d_df)
n_tp = len(top_true_positives_df)
n_fp = len(high_confidence_false_positives_df)
n_fn = len(high_confidence_false_negatives_df)
n_tn = len(top_true_negatives_df)

if phase3_2_comparison_df is not None:
    n_rescued = len(combined_rescued_genes_df)
    n_lost = len(protein_lost_after_integration_df)
    n_both_wrong = len(both_wrong_genes_df)
else:
    n_rescued = None
    n_lost = None
    n_both_wrong = None

summary_text = f"""
# Phase 4.0 — Biological Validation Preparation Summary

## Objective

This step prepared the final result tables and gene lists needed for biological validation of the T2D gene/protein association prediction framework.

The analysis used the official Phase 3 model:

- Combined Protein+Genomic SVM RBF
- Protein block: ProtBERT sliding-window embeddings
- Genomic block: K3/K4/Basic TSS-proximal regulatory features

## Final Master Comparison

A final master comparison table was created across major experimental phases:

- Protein handcrafted baseline
- Protein foundation embeddings
- Genomic handcrafted regulatory features
- Multimodal protein + genomic integration

The best overall ranking model was:

- Combined Protein+Genomic SVM RBF
- Test ROC-AUC: {official_metrics['roc_auc']:.4f}
- Test PR-AUC: {official_metrics['pr_auc']:.4f}
- Test F1: {official_metrics['f1']:.4f}
- Test MCC: {official_metrics['mcc']:.4f}

## Extracted Gene Lists

The following gene lists were extracted for biological validation:

- Top 50 combined predictions: {n_top50}
- True positives: {n_tp}
- High-confidence false positives: {n_fp}
- High-confidence false negatives: {n_fn}
- True negatives: {n_tn}
"""

if phase3_2_comparison_df is not None:
    summary_text += f"""

From Phase 3.2 error comparison:

- Combined-rescued genes: {n_rescued}
- Protein-correct but lost after integration: {n_lost}
- Both-wrong genes: {n_both_wrong}
"""

summary_text += """

## Recommended Biological Validation

The next step is to validate these gene lists using biological databases and enrichment tools.

Recommended databases/resources:

1. DisGeNET
2. Open Targets
3. GWAS Catalog
4. OMIM
5. KEGG
6. Reactome
7. Gene Ontology Biological Process
8. T2D/insulin/glucose/lipid/inflammation-related pathway gene sets

Recommended validation targets:

1. Top 50 combined predictions
2. Top true positives
3. Combined-rescued genes
4. High-confidence false positives
5. Both-wrong genes

The most important question is whether top-ranked and rescued genes are enriched for diabetes-relevant biological processes such as insulin secretion, glucose homeostasis, lipid metabolism, beta-cell function, inflammatory response, mitochondrial metabolism, PI3K-Akt signaling, AMPK signaling, and PPAR signaling.
"""

report_path = REPORT_DIR / "phase4_0_biological_validation_preparation_summary.md"

with open(report_path, "w") as f:
    f.write(summary_text)

print(summary_text)
safe_display_file(report_path)


# Phase 4.0 — Biological Validation Preparation Summary

## Objective

This step prepared the final result tables and gene lists needed for biological validation of the T2D gene/protein association prediction framework.

The analysis used the official Phase 3 model:

- Combined Protein+Genomic SVM RBF
- Protein block: ProtBERT sliding-window embeddings
- Genomic block: K3/K4/Basic TSS-proximal regulatory features

## Final Master Comparison

A final master comparison table was created across major experimental phases:

- Protein handcrafted baseline
- Protein foundation embeddings
- Genomic handcrafted regulatory features
- Multimodal protein + genomic integration

The best overall ranking model was:

- Combined Protein+Genomic SVM RBF
- Test ROC-AUC: 0.7290
- Test PR-AUC: 0.7573
- Test F1: 0.6590
- Test MCC: 0.3438

## Extracted Gene Lists

The following gene lists were extracted for biological validation:

- Top 50 combined predictions: 50
- True positives: 86
- High-confidence fals

In [17]:
# ============================================================
# LIST OUTPUT FILES
# ============================================================

print("=== RESULT FILES ===")
for p in sorted(RESULT_DIR.glob("*")):
    print(p)

print("\n=== GENE LIST FILES ===")
for p in sorted(GENE_LIST_DIR.glob("*")):
    print(p)

print("\n=== REPORT FILES ===")
for p in sorted(REPORT_DIR.glob("*")):
    print(p)

=== RESULT FILES ===
/content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0A_final_master_comparison_table.csv
/content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0A_ranking_by_mcc.csv
/content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0A_ranking_by_pr_auc.csv
/content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0A_ranking_by_roc_auc.csv
/content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_biological_validation_candidate_table.csv
/content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_gene_list_summary.csv
/content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_official_combined_svm_all_test_predictions.csv
/content/drive/MyDrive/Project_Protein/mod

In [18]:
display(master_comparison_df)
display(official_metrics_df)
display(top_50_predicted_t2d_df[["gene_id", "gene_symbol", "true_label", "combined_svm_score", "combined_error_type"]].head(50))
display(top_true_positives_df[["gene_id", "gene_symbol", "combined_svm_score"]].head(30))
display(high_confidence_false_positives_df[["gene_id", "gene_symbol", "combined_svm_score"]].head(30))

if phase3_2_comparison_df is not None:
    display(combined_rescued_genes_df[["gene_id", "gene_symbol", "true_label", "protein_svm_score", "combined_svm_score", "combined_error_type"]])

,phase,branch,representation,embedding_policy,feature_type,model,selection_policy,threshold_policy,test_roc_auc,test_pr_auc,test_f1,test_mcc,main_note
0,3.1,Multimodal,ProtBERT SW + K3/K4/Basic,early fusion direct concatenation,protein embedding + genomic handcrafted,SVM RBF,validation ROC-AUC,default_0.5,0.7290,0.7573,0.6590,0.3438,Official multimodal integration model
1,3.1,Protein shared split,ProtBERT SW,shared split,protein foundation embedding,SVM RBF,validation ROC-AUC,default_0.5,0.7274,0.7433,0.6667,0.3215,Best protein-only model on shared Phase 3 split
2,2.2,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,SVM RBF,diagnostic test comparison,default_0.5,0.6765,0.6504,0.6620,0.2933,"Diagnostic best genomic model, not official selection"
3,2.1,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,Random Forest,validation ROC-AUC,default_0.5,0.6496,0.6327,0.6406,0.2557,Official genomic handcrafted model
4,1.2E,Protein,ProtBERT,sliding_window_1022_stride_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6487,0.6551,0.5896,0.1941,Best protein ranking representation before shared split
5,1.2D,Protein,ProtBERT,truncated_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6371,0.6423,0.6014,0.1943,ProtBERT truncated
6,1.2B,Protein,ESM2_t6_8M,sliding_window_1022_stride_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.6277,0.6278,0.5870,0.1650,ESM-2 small sliding-window
7,1.2A,Protein,ESM2_t6_8M,truncated_1022,protein foundation embedding,Stacking,validation ROC-AUC,default_0.5,0.6202,0.6188,0.5926,0.1941,ESM-2 small truncated baseline
8,1.2C,Protein,ESM2_t12_35M,truncated_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.5875,0.5942,0.5654,0.0995,Larger ESM-2 did not improve under truncation
9,1.1,Protein,AAC + Physchem,handcrafted,handcrafted protein descriptors,Random Forest,validation ROC-AUC,default_0.5,0.5520,0.5550,0.5390,0.0480,Protein handcrafted baseline


,model,threshold,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Official Combined Protein+Genomic SVM RBF,0.5,0.671587,0.68254,0.637037,0.705882,0.659004,0.728976,0.757253,0.343763,96,40,49,86


,gene_id,gene_symbol,true_label,combined_svm_score,combined_error_type
0,ENSG00000053254,FOXN3,1,0.884630,TP
1,ENSG00000276644,DACH1,1,0.841229,TP
2,ENSG00000090266,NDUFB2,1,0.831156,TP
3,ENSG00000006468,ETV1,1,0.824098,TP
4,ENSG00000162992,NEUROD1,1,0.818462,TP
5,ENSG00000171105,INSR,1,0.817504,TP
6,ENSG00000128683,GAD1,1,0.813424,TP
7,ENSG00000107864,CPEB3,1,0.808824,TP
8,ENSG00000121297,TSHZ3,1,0.808711,TP
9,ENSG00000127334,DYRK2,1,0.805801,TP


,gene_id,gene_symbol,combined_svm_score
0,ENSG00000053254,FOXN3,0.884630
1,ENSG00000276644,DACH1,0.841229
2,ENSG00000090266,NDUFB2,0.831156
3,ENSG00000006468,ETV1,0.824098
4,ENSG00000162992,NEUROD1,0.818462
5,ENSG00000171105,INSR,0.817504
6,ENSG00000128683,GAD1,0.813424
7,ENSG00000107864,CPEB3,0.808824
8,ENSG00000121297,TSHZ3,0.808711
9,ENSG00000127334,DYRK2,0.805801


,gene_id,gene_symbol,combined_svm_score
22,ENSG00000160221,GATD3,0.770886
26,ENSG00000141314,RHBDL3,0.761844
34,ENSG00000108091,CCDC6,0.742967
37,ENSG00000196277,GRM7,0.732043
40,ENSG00000157212,PAXIP1,0.723585
41,ENSG00000135930,EIF4E2,0.718188
44,ENSG00000146267,FAXC,0.711850
54,ENSG00000105135,ILVBL,0.692967
56,ENSG00000104833,TUBB4A,0.691734
58,ENSG00000172594,SMPDL3A,0.690366


,gene_id,gene_symbol,true_label,protein_svm_score,combined_svm_score,combined_error_type
115,ENSG00000151883,PARP8,1,0.457089,0.540746,TP
112,ENSG00000006071,ABCC8,1,0.469028,0.522492,TP
29,ENSG00000134640,MTNR1B,1,0.494592,0.500000,TP
76,ENSG00000103978,TMEM87A,0,0.617864,0.494235,TN
54,ENSG00000122550,KLHL7,0,0.714343,0.491790,TN
229,ENSG00000105738,SIPA1L3,0,0.597829,0.486927,TN
94,ENSG00000077327,SPAG6,0,0.556301,0.476395,TN
99,ENSG00000197530,MIB2,0,0.576509,0.463581,TN
107,ENSG00000154832,CXXC1,0,0.607166,0.460217,TN
6,ENSG00000173320,STOX2,0,0.569132,0.452565,TN
